<h1 style="display:flex; align-items:center; line-height:1; margin:0;">
  <img src="rwcircle.png" style="height:36px; width:36px; border-radius:50%; flex-shrink:0;">
  <span style="padding-top:2px; margin-left:10px;"><b>Exploring Rocky Worlds DDT High Level Science Products (HLSPs) of the LHS 1140 system</b></span>
</h1>

***
**Author**: Nestor Espinoza (nespinoza@stsci.edu) | **Date of last change**: November 2026

# Summary
In this notebook, we explore the HLSPs of the LHS 1140 system. In particular, we explore JWST/MIRI time-series aimed at studying secondary eclipses of LHS 1140 b (the planet), and HST time-series aimed at studying LHS 1140 (the star) via the Rocky Worlds DDT program. Some of the JWST observations have evident stellar flare events present in the time-series --- which we aim to explore together with the HST ones in this Notebook. 

# Pre-study imports and definitions
First, let's load useful libraries for this notebook:

In [ ]:
# Generic python libraries:
import numpy as np
import matplotlib.pyplot as plt

# Some fits reading and time routines from astropy:
from astropy.io import fits
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation
import astropy.units as u

# Import h5py, which we will use to open h5 files:
import h5py

# This is a little of my own personal taste for plotting:
import seaborn as sns

# Import Observations to retrieve HLSPs (and observations too!):
from astroquery.mast import Observations

Next, we'll define some plotting standards --- this is, of course, personal preference!

In [ ]:
# One of my favorite seaborn styles for plots:
sns.set_style('ticks')

# Also personal taste on plotting dimensions, etc.:
plt.rcParams.update({
    'figure.figsize': (10, 4),
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
})

We finally setup a flag in case this notebook wishes to be ran offline --- here "offline" meaning without querying from MAST (e.g., in case queries are too slow). To this end, we have stored some pre-downloaded HLSPs on November 2026 in this repository (note at the time you run this notebook, higher versions of the HLSPs might be available):

In [ ]:
run_offline = False

# 1. Getting the JWST HLSPs

The very first thing we will do is find the Rocky Worlds DDT JWST HLSPs for the target. To do this, let's take a peek at all the HLSPs for LHS 1140 b:

In [ ]:
if not run_offline:

    lhs1140b_jwst = Observations.query_criteria(provenance_name="rocky-worlds", target_name = 'LHS 1140 b')
    
    columns = [
        'target_name',
        'obs_id',
        'project',
        'filters',
        'instrument_name',
        'dataproduct_type',
        't_min',
        't_max',
        't_exptime'
    ]
    
    display( lhs1140b_jwst[columns] )

Now let's figure out what data products there are available:

In [ ]:
if not run_offline:
    
    jwst_hlsp = Observations.get_product_list(lhs1140b_jwst)
    display( jwst_hlsp )

We will just be playing with the lightcurves --- and in particular, with the `h5` files. So let's get only those:

In [ ]:
if not run_offline: 

    jwst_hlsp_to_download = jwst_hlsp[ 
                                        [uri.endswith('_lc.h5') for uri in jwst_hlsp['dataURI']] 
                                     ]

Let's check:

In [ ]:
if not run_offline:

    display( jwst_hlsp_to_download )

Excellent. Let's get them:

In [ ]:
if not run_offline:

    jwst_downloads = Observations.download_products(jwst_hlsp_to_download)

    # Store local paths directly from download object:
    jwst_local_path1 = jwst_downloads['Local Path'][0]
    jwst_local_path2 = jwst_downloads['Local Path'][1]

else:

    jwst_local_path1 = 'offline_data/part2/JWST/hlsp_rocky-worlds_jwst_miri_lhs1140b-ecl001_f1500w_v1.0.1_lc.h5'
    jwst_local_path2 = 'offline_data/part2/JWST/hlsp_rocky-worlds_jwst_miri_lhs1140b-ecl002_f1500w_v1.0_lc.h5'

At the time of writing of this notebooks, this simply downloaded two HLSPs. In the future there will be more! Let's extract the data of eclipse 1 and 2 from those HLSPs:

In [ ]:
jwst_lc1 = h5py.File(jwst_local_path1, 'r')
jwst_lc2 = h5py.File(jwst_local_path2, 'r')

In [ ]:
# Data for eclipse 1:
time1 = jwst_lc1['time'][:]
flux1 = jwst_lc1['rawFlux'][:][0]
flux_error1 = jwst_lc1['rawFluxErr'][:][0]

# Data for eclipse 2:
time2 = jwst_lc2['time'][:]
flux2 = jwst_lc2['rawFlux'][:][0]
flux_error2 = jwst_lc2['rawFluxErr'][:][0]

Let's plot those two datasets:

In [ ]:
for i, (time, flux, flux_error) in enumerate([ (time1, flux1, flux_error1), (time2, flux2, flux_error2) ], start=1):

    plt.title(r'JWST/MIRI 15 $\mu$m lightcurve of LHS 1140 b --- Eclipse {0}'.format(i), fontsize=16)

    thours = ( time - time[0] ) * 24
    plt.errorbar(thours, flux, flux_error, fmt='.', color = 'black', alpha = 0.5)

    plt.xlabel('Time from exposure start (hours since {0:.2f})'.format(time[0]))
    plt.ylabel('Relative flux')

    plt.xlim(np.min(thours), np.max(thours))
    plt.show()

Beautiful two flares! Let's now jump to the HST data.

# 2. Getting the HST HLSPs

All right, let's now get the HST data. Let's repeat the process, now for HST:

In [ ]:
if not run_offline:

    lhs1140_hst = Observations.query_criteria(provenance_name="rocky-worlds", target_name = 'LHS 1140')
    display( lhs1140_hst[columns] )

In [ ]:
if not run_offline:

    hst_hlsp = Observations.get_product_list(lhs1140_hst)
    display( hst_hlsp )

Ah! Plenty of data products, including some `*_lc.fits` files ---the HST lightcurve HLSPs we want. Note from the filters or the filenames, one can identify that there are COS and STIS HLSPs. From here, there are three possible strategies to figure out which of all of those might overlap with our JWST lightcurves:

1. Take a look at the [Rocky Worlds DDT Status webpage](https://rockyworlds.stsci.edu/rw-website-schedule.html#details), and check if any of the JWST observations has an overlapping HST observations. 
2. Query the HST observations of proposal ID 18222 (listed above as the source of all those HLSPs), and from start and end times of _those_ figure out the overlap with JWST observations.
3. Simply download all the HST lightcurve HLSPs and figure out which ones match our JWST timestamps.

By eye, the easiest might be 1. If one goes to the [Rocky Worlds DDT Status webpage](https://rockyworlds.stsci.edu/rw-website-schedule.html#details) webpage, by eye it seems JWST Eclipse 1 had an HST observation contemporaneous with HST/COS visit 03, and JWST Eclipse 2 had an HST observation contemporaneous with HST/COs visit 04. Let's download those HLSPs:

In [ ]:
if not run_offline:

    hst_hlsp_to_download = hst_hlsp[
                                        [
                                            uri.endswith('_lc.fits') and
                                            '_cos_' in uri and
                                            ('visit03' in uri or 'visit04' in uri)
                                            for uri in hst_hlsp['dataURI']
                                        ]
                                   ]

    display( hst_hlsp_to_download )

Excellent! Let's get them:

In [ ]:
if not run_offline:
        
    hst_downloads = Observations.download_products(hst_hlsp_to_download)
    
    # Store local paths directly from download object:
    hst_local_path3 = hst_downloads['Local Path'][0]
    hst_local_path4 = hst_downloads['Local Path'][1]

else:

    hst_local_path3 = 'offline_data/part2/HST/hlsp_rocky-worlds_hst_cos_lhs1140-visit03_g130m_v1.0_lc.fits'
    hst_local_path4 = 'offline_data/part2/HST/hlsp_rocky-worlds_hst_cos_lhs1140-visit04_g130m_v1.0_lc.fits'

Let's open them up:

In [ ]:
visit3 = fits.open( hst_local_path3 )
visit4 = fits.open( hst_local_path4 )

Now, indexes 1 and 2 of these visit files are the ones that contain the lightcurves for segments A (1230-1365 Angstroms) and B (1140-1214 Angstroms) respectively. Note, however, that the units of time are in MJD UTC, as the header of index 0 notes:

In [ ]:
visit3[0].header

Because we want to be careful when comparing to JWST, we are going to convert timestamps from these products from MJD UTC to BJD TDB via `astropy`. To do so, first we write a little function that converts from MJD UTC to BJD TDB. Note that the datasets are matrices because they not only have the time samples, but also to which orbits they pertain, e.g.,

In [ ]:
visit3[1].data['TIME'].shape

Says we get 485 time samples for segment A in visit 3, which in turn has 3 orbits (consistent with [the observation description PDF of this program](https://www.stsci.edu/hst-program-info/download/hst/pdf/18222/)). Let's convert times from MJD UTC to BJD TDB (we'll assume HST is at Earth --- this adds a tiny offset that you might want to correct on real science applications, although its small --- of order a few tens of milliseconds). Let's create a mini function that does this:

In [ ]:
def mjdutc_to_bjdtdb(visit):

    # Extract target coordinates:
    target = SkyCoord(
                        ra=visit.header['RA_TARG'] * u.deg,
                        dec=visit.header['DEC_TARG'] * u.deg
                     )

    # Simple target location on Earth:
    location = EarthLocation.from_geocentric(0, 0, 0, unit=u.m)

    # Input timestamps in the data product:
    output_times = np.zeros( visit.data['TIME'].shape )
    for i in range( visit.data['TIME'].shape[0] ):

            # Let's pre-mask nans from non-nans:
            idx_nan = np.isnan( visit.data['TIME'][i, :] ) 
            idx_NOT_nan = ~idx_nan
        
            # Get UTC time in astropy format:
            time_utc = Time(
                                visit.data['TIME'][i, :][idx_NOT_nan],
                                format='mjd',
                                scale='utc',
                                location=location
                           )

            # Light travel time correction:
            ltt = time_utc.light_travel_time(target, kind='barycentric')

            # Get BJD TDB:
            output_times[i, idx_NOT_nan] = (time_utc.tdb + ltt).jd
            output_times[i, idx_nan] = np.nan

    return output_times

And now let's extract times, fluxes and errors using this function for the former:

In [ ]:
# Visit 03 dataset for both segments:
visit3_timesA = mjdutc_to_bjdtdb(visit3[1])
visit3_fluxA = visit3[1].data['FLUX']
visit3_fluxerrA = visit3[1].data['FLUXERROR']

visit3_timesB = mjdutc_to_bjdtdb(visit3[2])
visit3_fluxB = visit3[2].data['FLUX']
visit3_fluxerrB = visit3[2].data['FLUXERROR']

# Visit 04 dataset for both segments:
visit4_timesA = mjdutc_to_bjdtdb(visit4[1])
visit4_fluxA = visit4[1].data['FLUX']
visit4_fluxerrA = visit4[1].data['FLUXERROR']

visit4_timesB = mjdutc_to_bjdtdb(visit4[2])
visit4_fluxB = visit4[2].data['FLUX']
visit4_fluxerrB = visit4[2].data['FLUXERROR']

In [ ]:
np.nanmin(visit3_timesA)

The corresponding two plots for those visits:

In [ ]:
set_legend = True
legendA = r'Segment A (1230-1365 $\AA$)'
legendB = r'Segment B (1140-1214 $\AA$)'

for i, (timeA, fluxA, flux_errorA, timeB, fluxB, flux_errorB) in enumerate([ (visit3_timesA, visit3_fluxA, visit3_fluxerrA, visit3_timesB, visit3_fluxB, visit3_fluxerrB), 
                                                                             (visit4_timesA, visit4_fluxA, visit4_fluxerrA, visit4_timesB, visit4_fluxB, visit4_fluxerrB)
                                                                           ], start=3):

    plt.title(r'HST/COS lightcurve of LHS 1140 --- Visit {0}'.format(i), fontsize=16)

    min_time = np.nanmin(timeA)

    for k in range(3):
        
        thoursA = ( timeA[k, :] - min_time ) * 24
        thoursB = ( timeB[k, :] - min_time ) * 24

        plt.errorbar(thoursA, fluxA[k, :]*1e11, flux_errorA[k, :]*1e11, fmt='.', color = 'orangered', alpha = 0.5, label = legendA)
        plt.errorbar(thoursB, fluxB[k, :]*1e11, flux_errorB[k, :]*1e11, fmt='.', color = 'cornflowerblue', alpha = 0.5, label = legendB)

        if set_legend:
            plt.legend()
            set_legend = False
            legendA = None
            legendB = None
    
    plt.xlabel('Time from exposure start (hours since {0:.2f})'.format(min_time))
    plt.ylabel(r'HST/COS flux ($x 10^{-11}$ erg/s/cm$^{2}/\AA$)')

    plt.xlim(np.nanmin(timeA[0,:] - min_time) * 24, np.nanmax(timeA[-1,:] - min_time) * 24)
    plt.show()

All right! We are now ready to plot both side to side.

# 3. Combining HST and JWST datasets

Let's start with a combined plot of the JWST eclipse 1 dataset with HST visit 03:

In [ ]:
fig, ax1 = plt.subplots()

ax2 = ax1.twinx()

plt.title(r'JWST/MIRI + HST/COS lightcurve of LHS 1140 (Eclipse 1/visit 03)', fontsize=16)

tzero = time1[0]
thours_jwst = (time1 - tzero) * 24

# JWST — left axis
ax1.errorbar(thours_jwst, flux1, flux_error1,
             fmt='.', color='black', alpha=0.5)

# HST — right axis
for orbit in range(3):

    ax2.errorbar((visit3_timesA[orbit, :] - tzero) * 24,
                 visit3_fluxA[orbit, :]*1e11, visit3_fluxerrA[orbit, :]*1e11,
                 fmt='.', color='orangered', alpha=0.5)

    ax2.errorbar((visit3_timesB[orbit, :] - tzero) * 24,
                 visit3_fluxB[orbit, :]*1e11, visit3_fluxerrB[orbit, :]*1e11,
                 fmt='.', color='cornflowerblue', alpha=0.5)

ax1.set_xlabel('Time from exposure start (hours since {0:.2f})'.format(tzero))

ax1.set_ylabel('JWST/MIRI relative flux')
ax2.set_ylabel(r'HST/COS flux ($x 10^{-11}$ erg/s/cm$^{2}/\AA$)')

plt.show()

Let's do some close-ups:

In [ ]:
fig, ax1 = plt.subplots()

ax2 = ax1.twinx()

plt.title(r'JWST/MIRI + HST/COS lightcurve of LHS 1140 (Eclipse 1/visit 03)', fontsize=16)

tzero = time1[0]
thours_jwst = (time1 - tzero) * 24

# JWST — left axis
ax1.errorbar(thours_jwst, flux1, flux_error1,
             fmt='.', color='black', alpha=0.5)

# HST — right axis
for orbit in range(3):

    ax2.errorbar((visit3_timesA[orbit, :] - tzero) * 24,
                 visit3_fluxA[orbit, :]*1e11, visit3_fluxerrA[orbit, :]*1e11,
                 fmt='.', color='orangered', alpha=0.5)

    ax2.errorbar((visit3_timesB[orbit, :] - tzero) * 24,
                 visit3_fluxB[orbit, :]*1e11, visit3_fluxerrB[orbit, :]*1e11,
                 fmt='.', color='cornflowerblue', alpha=0.5)

ax1.set_xlabel('Time from exposure start (hours since {0:.2f})'.format(tzero))

ax1.set_ylabel('JWST/MIRI relative flux')
ax2.set_ylabel(r'HST/COS flux ($x 10^{-11}$ erg/s/cm$^{2}/\AA$)')

ax1.set_xlim(2.1, 2.9)
plt.show()

Same to the third orbit:

In [ ]:
fig, ax1 = plt.subplots()

ax2 = ax1.twinx()

plt.title(r'JWST/MIRI + HST/COS lightcurve of LHS 1140 (Eclipse 1/visit 03)', fontsize=16)

tzero = time1[0]
thours_jwst = (time1 - tzero) * 24

# JWST — left axis
ax1.errorbar(thours_jwst, flux1, flux_error1,
             fmt='.', color='black', alpha=0.5)

# HST — right axis
for orbit in range(3):

    ax2.errorbar((visit3_timesA[orbit, :] - tzero) * 24,
                 visit3_fluxA[orbit, :]*1e11, visit3_fluxerrA[orbit, :]*1e11,
                 fmt='.', color='orangered', alpha=0.5)

    ax2.errorbar((visit3_timesB[orbit, :] - tzero) * 24,
                 visit3_fluxB[orbit, :]*1e11, visit3_fluxerrB[orbit, :]*1e11,
                 fmt='.', color='cornflowerblue', alpha=0.5)

ax1.set_xlabel('Time from exposure start (hours since {0:.2f})'.format(tzero))

ax1.set_ylabel('JWST/MIRI relative flux')
ax2.set_ylabel(r'HST/COS flux ($x 10^{-11}$ erg/s/cm$^{2}/\AA$)')

ax1.set_xlim(4., 4.75)
plt.show()

Super interesting! Flare goes up in COS, but then down --- MIRI stays up. Ah, astrophysical misteries!

What about the second eclipse and visit 4?

In [ ]:
fig, ax1 = plt.subplots()

ax2 = ax1.twinx()

plt.title(r'JWST/MIRI + HST/COS lightcurve of LHS 1140 (Eclipse 2/visit 04)', fontsize=16)

tzero = time2[0]
thours_jwst = (time2 - tzero) * 24

# JWST — left axis
ax1.errorbar(thours_jwst, flux2, flux_error2,
             fmt='.', color='black', alpha=0.5)

# HST — right axis
for orbit in range(3):

    ax2.errorbar((visit4_timesA[orbit, :] - tzero) * 24,
                 visit4_fluxA[orbit, :]*1e11, visit4_fluxerrA[orbit, :]*1e11,
                 fmt='.', color='orangered', alpha=0.5)

    ax2.errorbar((visit4_timesB[orbit, :] - tzero) * 24,
                 visit4_fluxB[orbit, :]*1e11, visit4_fluxerrB[orbit, :]*1e11,
                 fmt='.', color='cornflowerblue', alpha=0.5)

ax1.set_xlabel('Time from exposure start (hours since {0:.2f})'.format(tzero))

ax1.set_ylabel('JWST/MIRI relative flux')
ax2.set_ylabel(r'HST/COS flux ($x 10^{-11}$ erg/s/cm$^{2}/\AA$)')

plt.show()

Ah! HST missed the event by a few hours. A bit of overlap with JWST, but nothing on the flare. 

***

<table style="width:100%; border:0;">
<tr>
<td style="text-align:left; vertical-align:middle;">
<a href="#top">Top of Page</a>
</td>

<td style="text-align:right; vertical-align:middle;">
<img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/7f391657bc2bf47a784400d65d2882703787416d/stsci_logo2.png"
     alt="Space Telescope Logo"
     width="80px"/>
</td>
</tr>
</table>